# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MUKUL-TIWARI/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## Ranked actions

The ranked queue prioritizes pages with high search visibility, weak ranking position, and little engagement.

Reason code:

- visible_low_engagement

Action:

- REVIEW_CONTENT

In [10]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn


In [11]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN=os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN=userdata.get("HF_TOKEN")
    except:
        pass

HF_TOKEN=HF_TOKEN or getpass.getpass("HF Token: ")

con=duckdb.connect()

con.execute(
f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL="hf://datasets/FlyRank/internship-warehouse"

TABLES={
"fact_daily":f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
}

query=f"""
SELECT
client_hash_id,
content_hash_id,
gsc_impressions,
gsc_clicks,
gsc_avg_position,
COALESCE(ga4_engaged_sessions,0) AS ga4_engaged_sessions
FROM {TABLES["fact_daily"]}
WHERE month='2026-03'
AND gsc_impressions>=100
LIMIT 300000
"""

df=con.sql(query).df()

df["score"]=(
df["gsc_impressions"]
+
np.where(df["gsc_avg_position"]>20,100,0)
+
np.where(df["ga4_engaged_sessions"]==0,100,0)
)

df["reason_code"]="visible_low_engagement"
df["action"]="REVIEW_CONTENT"

queue=df.sort_values("score",ascending=False)

queue.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,score,reason_code,action
10330,client_62f4a7e64f5e0096,content_34a70fea29d15f24,39003,2,2.764916,0,39103,visible_low_engagement,REVIEW_CONTENT
10359,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,0,8.613948,0,37468,visible_low_engagement,REVIEW_CONTENT
52000,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,28973,0,0.000311,0,29073,visible_low_engagement,REVIEW_CONTENT
39123,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,28947,0,0.002245,0,29047,visible_low_engagement,REVIEW_CONTENT
60086,client_62f4a7e64f5e0096,content_1642f339bd6e7c8d,24456,1,4.066814,0,24556,visible_low_engagement,REVIEW_CONTENT
53610,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,24233,0,0.317996,0,24333,visible_low_engagement,REVIEW_CONTENT
50761,client_73cda7b4e4f265ea,content_6a9c79f55413b447,21998,5,1.641740,0,22098,visible_low_engagement,REVIEW_CONTENT
259058,client_e547b89c05043229,content_757b1fa67827358d,19301,0,2.261644,0,19401,visible_low_engagement,REVIEW_CONTENT
10321,client_62f4a7e64f5e0096,content_60b99970e55b1ac5,16833,2,3.914157,0,16933,visible_low_engagement,REVIEW_CONTENT
226867,client_e547b89c05043229,content_eadb33b5df496f4a,16902,212,2.436990,9,16902,visible_low_engagement,REVIEW_CONTENT


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## Intended use

This ranked queue is designed to support editorial review by identifying pages that may benefit from content updates.

## Limits

The queue should be treated as decision support rather than automatic decision making. Search performance may also be influenced by seasonality, business priorities, or external events that are not represented in the model.

In [12]:
queue[["content_hash_id","score","reason_code","action"]].head(20)


,content_hash_id,score,reason_code,action
10330,content_34a70fea29d15f24,39103,visible_low_engagement,REVIEW_CONTENT
10359,content_945d6ff91386c817,37468,visible_low_engagement,REVIEW_CONTENT
52000,content_9c057b66c30a3abb,29073,visible_low_engagement,REVIEW_CONTENT
39123,content_9c057b66c30a3abb,29047,visible_low_engagement,REVIEW_CONTENT
60086,content_1642f339bd6e7c8d,24556,visible_low_engagement,REVIEW_CONTENT
53610,content_9c057b66c30a3abb,24333,visible_low_engagement,REVIEW_CONTENT
50761,content_6a9c79f55413b447,22098,visible_low_engagement,REVIEW_CONTENT
259058,content_757b1fa67827358d,19401,visible_low_engagement,REVIEW_CONTENT
10321,content_60b99970e55b1ac5,16933,visible_low_engagement,REVIEW_CONTENT
226867,content_eadb33b5df496f4a,16902,visible_low_engagement,REVIEW_CONTENT


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## Human review

Editors should verify search intent, content quality, business importance, and recent updates before taking action.

## Do not automate

- Content publishing
- Content deletion
- SEO strategy changes
- Business decisions
- Client communication

These actions require human approval.

In [13]:
queue.groupby("reason_code").size().reset_index(name="count")

,reason_code,count
0,visible_low_engagement,300000


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## Monitoring

Monitor ranking changes, engagement, and impressions after recommendations are applied.

Retrain the model when:

- New monthly warehouse data becomes available.
- Search behavior changes noticeably.
- Model performance decreases during validation.

In [14]:
queue.describe()


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,score
count,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000
mean,342.654717,1.043083,11.443093,0.029257,461.535050
std,482.677822,2.810937,12.233436,0.207527,485.656927
min,100.000000,0.000000,0.000000,0.000000,100.000000
25%,138.000000,0.000000,3.394715,0.000000,248.000000
50%,205.000000,0.000000,5.707865,0.000000,331.000000
75%,363.000000,1.000000,15.945588,0.000000,489.000000
max,39003.000000,274.000000,92.546154,18.000000,39103.000000


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Exports

The ranked action queue is exported for reuse in the capstone paper.

In [15]:
import os
import json

os.makedirs("work/outputs",exist_ok=True)

queue.to_csv(
"work/outputs/ranked_action_queue.csv",
index=False
)

metrics={
"rows":int(len(queue)),
"generated_month":"2026-03"
}

with open("work/outputs/playbook_metrics.json","w") as f:
    json.dump(metrics,f,indent=2)

print("Exports completed.")

queue.head()

Exports completed.


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,score,reason_code,action
10330,client_62f4a7e64f5e0096,content_34a70fea29d15f24,39003,2,2.764916,0,39103,visible_low_engagement,REVIEW_CONTENT
10359,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,0,8.613948,0,37468,visible_low_engagement,REVIEW_CONTENT
52000,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,28973,0,0.000311,0,29073,visible_low_engagement,REVIEW_CONTENT
39123,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,28947,0,0.002245,0,29047,visible_low_engagement,REVIEW_CONTENT
60086,client_62f4a7e64f5e0096,content_1642f339bd6e7c8d,24456,1,4.066814,0,24556,visible_low_engagement,REVIEW_CONTENT


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.